# Part B - Low-Rank Adaptation (LoRA)
## TOTAL MARKS: 50
## Overview
### `Name: Muhammad Hassan Ibrahim`
### `Roll Number: 28100395`

References:
https://magazine.sebastianraschka.com/p/lora-and-dora-from-scratch


## Instructions

<table style="width:100%; table-layout:fixed;">
<tr>

<td style="vertical-align:top; padding:12px; width:33%;">
  <h3 style="margin:0 0 8px 0; font-weight:800; color:#e53935;">
    Plagiarism Policy
  </h3>
  <ul style="margin:6px 0 0 18px; padding:0; color:#e53935;">
    <li>
      <strong>All work must be done independently.</strong><br>
      Any plagiarism or cheating (from peers or the internet) will be immediately referred to the DC.
      If you are unsure what constitutes plagiarism, consult the TAs in a timely manner.
    </li>
    <li style="margin-top:8px;">
      <strong>Do not look at anyone else’s code.</strong>
    </li>
  </ul>
</td>

<td style="vertical-align:top; padding:12px; width:33%;">
  <h3 style="margin:0 0 8px 0; font-weight:800; color:#e53935;">
    Submission Instructions
  </h3>
  <ul style="margin:6px 0 0 18px; padding:0; color:#e53935;">
    <li>
      Submit <strong>all</strong> required <code>.ipynb</code> and <code>.py</code> files on LMS
      (search how to extract scripts if confused).
    </li>
    <li style="margin-top:8px;">
      <strong>Submissions via Dropbox or email will not be accepted.</strong>
    </li>
    <li style="margin-top:8px;">
      Zip your files as <code>RollNumber_PAx.zip</code> and ensure your roll number is filled in correctly.
    </li>
    <li style="margin-top:8px;">
      <strong>Deviation from the naming convention will result in a penalty.</strong>
    </li>
    <li style="margin-top:8px;">
      <strong>Expected file structure:</strong>
    </li>
  </ul>
  <pre style="margin:8px 0 0 0; font-size:90%; color:#e53935;">
26100076_PA3.zip
├─ 26100076_PA3.ipynb
└─ 26100076_PA3.py
  </pre>
</td>

<td style="vertical-align:top; padding:12px; width:33%;">
  <h3 style="margin:0 0 8px 0; font-weight:800; color:#e53935;">
    General Instructions
  </h3>
  <ul style="margin:6px 0 0 18px; padding:0; color:#e53935;">
    <li>
      <strong>Ensure all cells are executed before submission.</strong>
    </li>
    <li style="margin-top:8px;">
      <strong>Do not remove or modify any pre-written code.</strong>
    </li>
    <li style="margin-top:8px;">
      <strong>All parts of the assignment must be attempted.</strong>
    </li>
  </ul>
</td>

</tr>
</table>

# Low-Rank Adaptation (LoRA) — An Intuitive Explanation

Large language models (LLMs) and vision transformers contain **millions or billions of parameters** (weights).  
Fine-tuning these models normally requires updating *all* of these weights, which is computationally expensive and often limited by GPU memory.

---

### Regular Fine-Tuning

In standard training or fine-tuning:

- Each layer has a large weight matrix **W**
- Training learns a full update matrix **ΔW**
- The updated weights are:

$$
W_{\text{updated}} = W + \Delta W
$$

**Drawback:**  
The update matrix **ΔW** is very large, making training slow and memory-intensive.

---

### LoRA: The Key Idea

LoRA (Low-Rank Adaptation) is based on the observation that:

> Most useful weight updates can be represented by **simple patterns**, rather than a full, complex matrix.

Instead of learning the full **ΔW**, LoRA **approximates it** using two much smaller matrices:

$$
\Delta W \approx A \cdot B
$$

where:
- **A** and **B** are small matrices
- Their matrix product has the same shape as **W**

The weight update becomes:

$$
W_{\text{updated}} = W + A \cdot B
$$

The figure below illustrates these formulas for full finetuning and LoRA side by side.

![LoRA illustration](LoRAImage.webp)

---

### Intuitive Explanation

Think of the model as a large machine with many adjustment knobs.

- **Full fine-tuning:**  
  Adjust every knob individually.

- **LoRA:**  
  Keep the original knobs fixed and add a small attachment that adjusts many knobs together in a coordinated way.

This attachment (matrices **A** and **B**) captures the most important changes while using far fewer parameters.

## Implementing a LoRA Layer (2.5 MARKS)
Implement a LoRA layer that adapts a neural network without directly modifying its original weights. Instead of learning a full weight update, represent the adaptation as a low-rank decomposition that captures task-specific changes using a compact set of parameters. This formulation reduces both memory usage and computational cost compared to full fine-tuning.

Design the LoRA layer using two trainable low-rank matrices whose product defines the weight update. Scale this update appropriately and optionally apply dropout to regulate its effect. During the forward pass, compute the low-rank update independently so it can later be combined with the output of a frozen base layer in a modular and efficient manner.

In [177]:
import torch
import torch.nn as nn
import torch.nn.functional as F


In [178]:
class LoRALayer(nn.Module):
    """
    Implements a LoRA low-rank adaptation layer.
    """

    def __init__(self, in_features, out_features, rank=8, alpha=16, dropout=0.0):
        super().__init__()
        # TODO: store rank and alpha
        self.rank = rank
        self.alpha = alpha 
        self.in_features = in_features
        self.out_features = out_features
        self.dropout = nn.Dropout(p = dropout)
        # TODO: initialize A and B matrices
        self.matrix_A = nn.Parameter(torch.randn(rank , in_features))
        self.matrix_B = nn.Parameter(torch.zeros(out_features , rank))
        # TODO: initialize dropout

    def forward(self, x):
        # TODO: apply dropout
        X_dropped = self.dropout(x)
        # TODO: compute low-rank update 
        update = X_dropped @ (self.matrix_A.T)
        update = update @ self.matrix_B.T
        # TODO: scale and return
        scale = self.alpha / self.rank
        update = update * scale

        return update


## LoRA-Adapted Linear Layer (non-merged) (5 MARKS)
Implement a LoRA-adapted version of a linear layer by wrapping an existing nn.Linear module. The original weights and bias should be frozen to ensure that only the low-rank adaptation is trained. The forward pass should combine the output of the frozen linear layer with the output of the LoRA layer, illustrating how task-specific updates can be added without modifying the base parameters.

In [179]:
class LoRALinear(nn.Module):
    """
    A linear layer with LoRA adaptation.
    """

    def __init__(self, linear_layer, rank=8, alpha=16, dropout=0.0):
        super().__init__()
        # TODO: store linear layer
        self.base_layer = linear_layer
        # TODO: freeze base weights
        for parameter in self.base_layer.parameters():
            parameter.requires_grad = False
        # TODO: create LoRALayer
        in_features = linear_layer.in_features
        out_features = linear_layer.out_features
        self.LoRA_Layer = LoRALayer(in_features , out_features , rank , alpha , dropout)

    def forward(self, x):
        # TODO: compute base output
        base_output = self.base_layer(x)
        # TODO: compute LoRA output
        LoRA_output = self.LoRA_Layer(x)
        # TODO: return combined output
        return base_output + LoRA_output
        


## Define a Small Network with One Linear Layer (2.5 MARKS)
Create a simple experimental setup using a single linear layer and a randomly generated input tensor. Fix the random seed to ensure reproducibility. Compute and record the output of the original linear layer, which will serve as a baseline for comparison when LoRA is introduced.

In [180]:
# TODO: set random seed
random_seed = 42
torch.manual_seed(random_seed)

# TODO: create linear layer
layer =  nn.Linear(32 , 64)
# TODO: create input tensor
input = torch.randn(24 , 32)
# TODO: compute and print baseline output
output = layer(input)
print(output)


tensor([[-0.4570, -0.8549,  1.2750,  ...,  0.5257, -0.1998, -0.3800],
        [-0.8695, -0.7507,  0.1197,  ..., -0.0659, -0.8964,  0.3407],
        [ 0.8915,  0.1417, -0.1918,  ...,  0.5087,  0.4744, -0.2521],
        ...,
        [ 0.3313,  0.5063, -0.2053,  ..., -0.2589, -0.0430,  0.2992],
        [ 0.0039,  0.4788,  0.8471,  ..., -0.3750,  1.3011,  0.4661],
        [ 0.0925,  0.3107, -0.4953,  ..., -0.1678,  0.1861, -0.4938]],
       grad_fn=<AddmmBackward0>)


## Apply LoRA to the Linear Layer (2.5 MARKS)
Apply the LoRA-adapted linear wrapper to the previously defined layer. Run the same input through this modified layer and observe the change in output compared to the baseline. This step demonstrates how LoRA introduces an additional learned transformation while keeping the original layer weights unchanged.

In [181]:
# TODO: wrap layer with LoRALinear
# TODO: compute and print LoRA output
layer_lora = LoRALinear(layer,rank=2,alpha=4)
LoRA_output = layer_lora(input)
print(LoRA_output)


tensor([[-0.4570, -0.8549,  1.2750,  ...,  0.5257, -0.1998, -0.3800],
        [-0.8695, -0.7507,  0.1197,  ..., -0.0659, -0.8964,  0.3407],
        [ 0.8915,  0.1417, -0.1918,  ...,  0.5087,  0.4744, -0.2521],
        ...,
        [ 0.3313,  0.5063, -0.2053,  ..., -0.2589, -0.0430,  0.2992],
        [ 0.0039,  0.4788,  0.8471,  ..., -0.3750,  1.3011,  0.4661],
        [ 0.0925,  0.3107, -0.4953,  ..., -0.1678,  0.1861, -0.4938]],
       grad_fn=<AddBackward0>)


## Merged LoRA Version (for inference) (5 MARKS)
Implement an alternative version of the linear layer in which the LoRA weight update is merged directly into the original weight matrix. In this formulation, compute the low-rank weight update and add it to the frozen base weights before applying the linear operation. This merged approach reflects how LoRA can be deployed efficiently during inference.

In [182]:
class LinearWithLoRAMerged(nn.Module):
    """
    Linear layer where LoRA weights are merged into W.
    """

    def __init__(self, linear, rank, alpha):
        super().__init__()
        # TODO: store linear layer
        self.linear_layer = linear
        self.rank = rank
        self.alpha = alpha
        # TODO: create LoRALayer
        self.LoRA_layer = LoRALinear(linear , rank , alpha , 0.0)


    def forward(self, x):
        # TODO: compute delta W
        matrix_A = self.LoRA_layer.LoRA_Layer.matrix_A
        matrix_B = self.LoRA_layer.LoRA_Layer.matrix_B
        scale = self.alpha/self.LoRA_layer.LoRA_Layer.rank
        delta_w = scale * (matrix_B @ matrix_A)
        # TODO: merge weights
        merged_W = self.linear_layer.weight + delta_w
        # TODO: apply linear op
        return F.linear(x, merged_W, self.linear_layer.bias)
        


## Verify Mathematical Equivalence  (0.5 MARKS)
Verify that the merged and non-merged LoRA implementations produce identical outputs when initialized with the same parameters. Ensure both models share the same low-rank matrices, then compare their outputs on the same input. Report the maximum absolute difference to confirm numerical equivalence between the two approaches.

In [183]:
# TODO: instantiate merged and unmerged models
# TODO: copy LoRA parameters
# TODO: compare outputs

layer_lora_unmerged = LoRALinear(layer, rank=2, alpha=4)
layer_lora_merged = LinearWithLoRAMerged(layer, rank=2, alpha=4)

layer_lora_merged.LoRA_layer.LoRA_Layer.matrix_A.data = layer_lora_unmerged.LoRA_Layer.matrix_A.data.clone()
layer_lora_merged.LoRA_layer.LoRA_Layer.matrix_B.data = layer_lora_unmerged.LoRA_Layer.matrix_B.data.clone()

out_unmerged = layer_lora_unmerged(input)
out_merged = layer_lora_merged(input)
diff = (out_unmerged - out_merged).abs().max()
print(f"Max absolute difference: {diff.item()}")



Max absolute difference: 0.0


# Applying LoRA Layers to LLM (2.5 MARKS)


## Preparing the Dataset
Prepare a text dataset suitable for causal language modeling. Load a subset of the AG News dataset, shuffle it for reproducibility, and split it into training and evaluation sets. Tokenize the text with fixed-length padding and truncation, and construct labels that mask padding tokens so they do not contribute to the training loss. Finally, format the dataset for PyTorch and create data loaders for batch-based training and evaluation.

In [184]:
%pip install transformers datasets

Note: you may need to restart the kernel to use updated packages.


In [185]:

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset


In [186]:
from datasets import load_dataset
from torch.utils.data import DataLoader

def prepare_dataset(tokenizer, max_samples=500, max_length=128):
    """
    Prepares train and test DataLoaders for language model fine-tuning.

    Args:
        tokenizer: HuggingFace tokenizer
        max_samples: number of training samples to use
        max_length: maximum sequence length

    Returns:
        train_loader, test_loader
    """
    dataset = load_dataset("ag_news")
    
    train_data = dataset["train"].shuffle(seed=42).select(range(max_samples))
    test_data  = dataset["test"].shuffle(seed=42).select(range(max_samples))

    def tokenize(sample):
        encoding = tokenizer(sample["text"] , truncation=True , max_length=max_length , padding="max_length")
        labels = encoding["input_ids"].copy()
        labels = [[-100 if token == tokenizer.pad_token_id else token for token in label] for label in labels]
        encoding["labels"] = labels
        return encoding
    
    train_data = train_data.map(tokenize, batched=True)
    test_data  = test_data.map(tokenize, batched=True)

    train_data.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
    test_data.set_format(type="torch",  columns=["input_ids", "attention_mask", "labels"])

    train_loader = DataLoader(train_data, batch_size=8, shuffle=True)
    test_loader  = DataLoader(test_data,  batch_size=8, shuffle=False)

    return train_loader, test_loader

## Initializing the Base Language Model
Load a pre-trained causal language model and its corresponding tokenizer. Configure the tokenizer to use a valid padding token and move the model to the appropriate computation device. Keep this base model in evaluation mode to serve as a frozen reference for comparison with the LoRA-adapted version.

In [187]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [188]:
# TODO: load tokenizer
# TODO: load base model
model_name = "EleutherAI/gpt-neo-125M"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_name)
# TODO: set device and eval mode
model.to(DEVICE)
model.eval()

Loading weights:   0%|          | 0/160 [00:00<?, ?it/s]

GPTNeoForCausalLM LOAD REPORT from: EleutherAI/gpt-neo-125M
Key                                                   | Status     |  | 
------------------------------------------------------+------------+--+-
transformer.h.{0...11}.attn.attention.masked_bias     | UNEXPECTED |  | 
transformer.h.{0, 2, 4, 6, 8, 10}.attn.attention.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


GPTNeoForCausalLM(
  (transformer): GPTNeoModel(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(2048, 768)
    (drop): Dropout(p=0.0, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPTNeoBlock(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPTNeoAttention(
          (attention): GPTNeoSelfAttention(
            (attn_dropout): Dropout(p=0.0, inplace=False)
            (resid_dropout): Dropout(p=0.0, inplace=False)
            (k_proj): Linear(in_features=768, out_features=768, bias=False)
            (v_proj): Linear(in_features=768, out_features=768, bias=False)
            (q_proj): Linear(in_features=768, out_features=768, bias=False)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPTNeoMLP(
          (c_fc): Linear(in_features=768, out_features=3072, bias=True)
          (c_proj): Linear(in_fe

## Creating a LoRA-Adapted Model (2.5 MARKS)
Instantiate a second copy of the pre-trained language model that will be modified using LoRA. This model will be used for training and should be set to training mode. Maintaining separate base and LoRA-adapted models allows direct comparison of parameter counts and downstream performance.

In [189]:
# TODO: load LoRA model
# TODO: set training mode
model_lora = AutoModelForCausalLM.from_pretrained(model_name)
apply_lora_to_model(model_lora, rank=8, alpha=16)
model_lora.to(DEVICE)
model_lora.train()

Loading weights:   0%|          | 0/160 [00:00<?, ?it/s]

GPTNeoForCausalLM LOAD REPORT from: EleutherAI/gpt-neo-125M
Key                                                   | Status     |  | 
------------------------------------------------------+------------+--+-
transformer.h.{0...11}.attn.attention.masked_bias     | UNEXPECTED |  | 
transformer.h.{0, 2, 4, 6, 8, 10}.attn.attention.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


GPTNeoForCausalLM(
  (transformer): GPTNeoModel(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(2048, 768)
    (drop): Dropout(p=0.0, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPTNeoBlock(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPTNeoAttention(
          (attention): GPTNeoSelfAttention(
            (attn_dropout): Dropout(p=0.0, inplace=False)
            (resid_dropout): Dropout(p=0.0, inplace=False)
            (k_proj): LoRALinear(
              (base_layer): Linear(in_features=768, out_features=768, bias=False)
              (LoRA_Layer): LoRALayer(
                (dropout): Dropout(p=0.0, inplace=False)
              )
            )
            (v_proj): LoRALinear(
              (base_layer): Linear(in_features=768, out_features=768, bias=False)
              (LoRA_Layer): LoRALayer(
                (dropout): Dropout(p=0.0, inplace=False)
              )
            )
            (q_proj): LoRALinear(
         

## Applying LoRA to Attention Layers (2.5 MARKS)
Modify the language model by replacing selected linear layers within the attention mechanism with LoRA-adapted linear layers. Target projection layers commonly used in self-attention, such as query, key, value, and output projections. This step injects low-rank adaptations into the model while leaving the original weights structurally intact.

In [190]:
def apply_lora_to_model(model, rank=8, alpha=16):
    """
    Replaces selected attention projection Linear layers with LoRALinear wrappers.

    Args:
        model: a transformer model (HuggingFace or similar)
        rank: LoRA rank r
        alpha: LoRA scaling alpha
        dropout: LoRA dropout (applied inside LoRALayer)

    Returns:
        model (modified in-place or returned for convenience)
    """
    # TODO: iterate modules
    # TODO: replace attention linear layer
    for name, module in model.named_modules():
        if "attn" in name:
                for layer_name in ["q_proj", "k_proj", "v_proj", "out_proj"]:
                    if hasattr(module, layer_name):
                        original_layer = getattr(module, layer_name)
                        setattr(module, layer_name, LoRALinear(original_layer, rank=rank, alpha=alpha))
    return model
            
     


## Freezing Base Parameters and Enabling LoRA Training (2.5 MARKS)
Freeze all original model parameters to prevent them from being updated during training. Enable gradient updates only for the LoRA low-rank matrices. This ensures that learning is restricted to the parameter-efficient LoRA components while the pre-trained knowledge of the base model is preserved.

In [191]:
# TODO: freeze base parameters
# TODO: enable LoRA parameters
for name, param in model_lora.named_parameters():
    if "matrix_A" in name or "matrix_B" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False


## Inspecting Trainable Parameters
Verify that only the intended LoRA parameters are trainable by inspecting the model’s parameter list. This step helps confirm that the parameter-freezing strategy has been applied correctly before training begins.

In [192]:
def check_trainable_params(model):
    # TODO: print trainable parameters
    for name, param in model.named_parameters():
        if param.requires_grad:
            print(f"{name}: {param.shape}")

check_trainable_params(model_lora)

transformer.h.0.attn.attention.k_proj.LoRA_Layer.matrix_A: torch.Size([8, 768])
transformer.h.0.attn.attention.k_proj.LoRA_Layer.matrix_B: torch.Size([768, 8])
transformer.h.0.attn.attention.v_proj.LoRA_Layer.matrix_A: torch.Size([8, 768])
transformer.h.0.attn.attention.v_proj.LoRA_Layer.matrix_B: torch.Size([768, 8])
transformer.h.0.attn.attention.q_proj.LoRA_Layer.matrix_A: torch.Size([8, 768])
transformer.h.0.attn.attention.q_proj.LoRA_Layer.matrix_B: torch.Size([768, 8])
transformer.h.0.attn.attention.out_proj.LoRA_Layer.matrix_A: torch.Size([8, 768])
transformer.h.0.attn.attention.out_proj.LoRA_Layer.matrix_B: torch.Size([768, 8])
transformer.h.1.attn.attention.k_proj.LoRA_Layer.matrix_A: torch.Size([8, 768])
transformer.h.1.attn.attention.k_proj.LoRA_Layer.matrix_B: torch.Size([768, 8])
transformer.h.1.attn.attention.v_proj.LoRA_Layer.matrix_A: torch.Size([8, 768])
transformer.h.1.attn.attention.v_proj.LoRA_Layer.matrix_B: torch.Size([768, 8])
transformer.h.1.attn.attention.q_pro

## Comparing Parameter Counts  (2.5 MARKS)
Compute and compare the total number of parameters and the number of trainable parameters for both the base model and the LoRA-adapted model. Report the percentage reduction in trainable parameters achieved through LoRA to quantify its efficiency benefits.

In [193]:
def count_parameters(model):
    # TODO: compute parameter counts
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    reduction = 100 * (1 - trainable / total)
    
    print(f"Total parameters:     {total:,}")
    print(f"Trainable parameters: {trainable:,}")
    print(f"Reduction:            {reduction:.2f}%")
    
    return total, trainable
count_parameters(model)  
count_parameters(model_lora)

Total parameters:     125,198,592
Trainable parameters: 125,198,592
Reduction:            0.00%
Total parameters:     125,788,416
Trainable parameters: 589,824
Reduction:            99.53%


(125788416, 589824)

## Training the LoRA-Adapted Model  (2.5 MARKS)
Train the LoRA-adapted model using the prepared dataset and an appropriate optimizer. Perform multiple training epochs while monitoring the training loss. Since only a small subset of parameters is being updated, this training process is significantly more efficient than full fine-tuning.

In [194]:
# TODO: training loop
model_lora.to(DEVICE)
train_loader, test_loader = prepare_dataset(tokenizer)
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model_lora.parameters()), lr=1e-4)

model_lora.train()

for epoch in range(2):
    total_loss = 0
    for step, batch in enumerate(train_loader):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        optimizer.zero_grad()
        outputs = model_lora(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1} | Loss: {total_loss / len(train_loader):.4f}")


## DO NOT FORGET TO PRINT YOUR LOSS

Epoch 1 | Loss: 4.1088
Epoch 2 | Loss: 3.8207


## Evaluating Model Accuracy  (5 MARKS)
Evaluate both the original base model and the LoRA-adapted model on the test dataset. Compute token-level accuracy while ignoring masked positions. Compare the results to assess how well the LoRA-based fine-tuning performs relative to the frozen pre-trained model.

In [195]:
@torch.no_grad()
def compute_accuracy(model, dataloader, device):
    """
    TODO:
    1. Disable gradients and set model to eval mode
    2. Get model logits for each batch
    3. Compute predictions using argmax
    4. Mask out padding tokens (-100)
    5. Return token-level accuracy (%)
    """
    model.eval()
    total_correct = 0
    total_tokens = 0

    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits 
        shift_logits = logits[:, :-1, :]
        shift_labels = labels[:, 1:]
        predictions = shift_logits.argmax(dim=-1)
        mask = shift_labels != -100
        total_correct += (predictions[mask] == shift_labels[mask]).sum().item()
        total_tokens += mask.sum().item()
    return 100 * (total_correct / total_tokens)


In [196]:
print(f"Test accuracy orig model: {compute_accuracy(model, test_loader, DEVICE):.2f}%")
print(f"Test accuracy LoRA model: {compute_accuracy(model_lora, test_loader, DEVICE):.2f}%")

Test accuracy orig model: 30.44%
Test accuracy LoRA model: 34.37%


# Reflective Questions (12 Marks)
Answer the reflective questions provided below.
Q1. Why is representing weight updates as a low-rank decomposition effective for large language models? \
Ans. All LLMs are over-parameterized. Fine-tuning a model actually depends on low intrinsic dimensions, we can make the necessary changes to a model and tine tune it by updating less than 1% of the parameters which reduces memory overhead and prevents catastrophic forgetting. Updating all the billions of parameters is extremely ineffecient and unnecesary. \
Q2. What assumptions does LoRA make about the nature of task-specific adaptations?\
Ans. LoRA assumed task specific adaption is extremely simple and can be done by updating very few parameters instead of billions.\
Q3. Why are LoRA layers typically applied to attention projection layers rather than all linear layers?\
Ans. Attention layers is where all the reasoning logic is implemented the attention layers is what the transformer models relationships between words, we could apply it to the linear layers but it is unnecesary.\
Q4. How does the choice of rank affect the expressive capacity of the LoRA adaptation?\
Ans. higher ranks have better expressive capacity which allows the model to learn more complex relations but it also often leads to overfitting and requires more memory. Lower rank models often have the same accuracy but use lesser memory and dont lead to overfitting or catastrophic forgetting. \
Q5. Why might a LoRA-adapted model perform comparably to a fully fine-tuned model on some tasks?\
Ans.The pre traine weights of the base model contain most of the information the model needs the low rank LoRA models only give a slight nudge which is comparable to the change in weights cause by full fine tuning so both training methods lead to similar final weights and thus similar accuracy. \
Q6. Under what conditions could LoRA lead to worse performance than full fine-tuning?
Ans. LoRA can lead to lower performance if the rank used is way too low to capture complex relations or if we use insufficient training data or the task needed to be performed if completely different to the data the base model was trained on so the model needs major changes in weights to be able to perform the tasks.